In [1]:
import glob
import geopandas as gpd

grid_path = 'data/*.parquet'
grids_list = glob.glob(grid_path)

expected_columns = {'geometry', 'grid_100000_id', 'slope', 'dou', 'ent_5',
                    'ent_10', 'ent_15', 'ndvi', 'population', 'green_area',
                    'pub_trans_count', 'street_walk_length', 'num_street_intersections'}

missing = {}
for path in grids_list:
    gdf = gpd.read_parquet(path)
    absent = expected_columns - set(gdf.columns)
    if absent:
        missing[path] = absent

if missing:
    print(f"{len(missing)} files missing columns:\n")
    for path, cols in missing.items():
        print(f"  {path}: {cols}")
else:
    print("All files have the expected columns.")

2 files missing columns:

  data/grids_100_830.parquet: {'pub_trans_count'}
  data/grids_100_850.parquet: {'green_area', 'pub_trans_count'}


In [3]:
import time, duckdb

path = "data/grids_100_570.parquet"   # one of your files

con = duckdb.connect()
con.execute("SET threads=4; SET memory_limit='10GB';")

t0 = time.perf_counter()
summary = con.execute(f"SUMMARIZE SELECT * FROM read_parquet('{path}')").df()
elapsed = time.perf_counter() - t0

print(summary)
print(f"\n1 file: {elapsed:.2f}s  ->  est. 60 files: {elapsed*60/60:.1f}s to {elapsed*60:.0f}s")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                 column_name  \
0                      index   
1                   geometry   
2             grid_100000_id   
3                      slope   
4                        dou   
5                      ent_5   
6                     ent_10   
7                     ent_15   
8                       ndvi   
9                 population   
10                green_area   
11           pub_trans_count   
12        street_walk_length   
13  num_street_intersections   

                                          column_type  \
0                                              BIGINT   
1   GEOMETRY('{"$schema":"https://proj.org/schemas...   
2                                              BIGINT   
3                                               FLOAT   
4                                            SMALLINT   
5                                               FLOAT   
6                                               FLOAT   
7                                               FLOAT   
8     

In [4]:
import duckdb
con = duckdb.connect()
con.execute("SET threads=4; SET memory_limit='10GB';")  # match your Slurm request
df = con.execute("SUMMARIZE SELECT * FROM read_parquet('data/*.parquet')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
import pandas as pd

# DuckDB returns most stats as text, so convert to numbers where possible
num_cols = ["min", "max", "avg", "std", "q25", "q50", "q75"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.rename(columns={
    "column_name": "column",
    "column_type": "type",
    "approx_unique": "~unique",
    "null_percentage": "null_%",
})

df["null_%"] = df["null_%"].astype(float).round(2)
df["count"] = df["count"].map("{:,}".format)
df["~unique"] = df["~unique"].map("{:,}".format)

df = df[["column", "type", "count", "null_%", "~unique",
         "min", "q25", "q50", "q75", "max", "avg", "std"]]

# Print the full table without truncation, with sensible number formatting
with pd.option_context("display.max_columns", None,
                       "display.width", 200,
                       "display.float_format", "{:,.3f}".format):
    print(df.to_string(index=False))

df.to_csv("summary.csv", index=False)

                  column                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                